# 🎧 Audiolibros para estudiar — V3 con voz natural

**Una unidad PDF → una muestra → un audiolibro MP3 completo.** No hace resúmenes: lee el texto extraíble página por página.

**Cómo usar (una vez abierto en Google Colab):**

1. Pulsa ▶ **Preparar** (primera vez puede tardar varios minutos; se descargarán modelos de voz).
2. Pulsa ▶ **Subir PDF** y selecciona tu unidad de 10–15 páginas.
3. Pulsa ▶ **Escuchar muestra**. Puedes cambiar la voz y volver a ejecutarlo.
4. **Solo si te gusta**, pulsa ▶ **Convertir toda la unidad** para obtener tu MP3.

**Gratis por página:** Kokoro es un modelo de voz abierto. Google Colab tiene límites de uso, disponibilidad y recursos; no garantiza ejecución permanente. Se requiere conexión para instalar y descargar el modelo. Para PDFs con texto escaneado/fotos no hay OCR todavía. Las tablas, fórmulas y archivos en columnas pueden leerse en orden incorrecto. Comparte grabaciones únicamente si tienes permiso para distribuir el material.

**Importante:** aquí la voz robótica no se activa como respaldo. Si Kokoro falla, te muestra el error en lugar de darte un audio feo y decir que salió bien.

Proyecto de voz: https://github.com/hexgrad/kokoro (licencia Apache-2.0).

In [ ]:
# @title PASO 1 · Preparar voz natural y lector PDF (pulsa ▶)
import subprocess, sys, shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kokoro>=0.9.4', 'PyMuPDF>=1.24', 'soundfile', 'numpy'], check=True)
if shutil.which('espeak-ng') is None or shutil.which('ffmpeg') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'espeak-ng', 'ffmpeg'], check=True)
print('✅ Instalación lista. Continúa al PASO 2.')

In [ ]:
# @title PASO 2 · Subir la unidad en PDF (pulsa ▶)
import io, re, unicodedata
from pathlib import Path
import fitz
from google.colab import files

print('📄 Selecciona UNA unidad PDF de aproximadamente 10 a 15 páginas.')
subidos = files.upload()
nombres_pdf = [n for n in subidos if n.lower().endswith('.pdf')]
if len(nombres_pdf) != 1:
    raise ValueError('Por favor, sube un único PDF. Repite el paso 2 y elige solo una unidad.')
MUESTRA_OK = None  # exige volver a escuchar la muestra con cada PDF nuevo
NOMBRE_PDF = nombres_pdf[0]
DATOS_PDF = subidos[NOMBRE_PDF]
doc = fitz.open(stream=DATOS_PDF, filetype='pdf')
if doc.needs_pass:
    doc.close()
    raise ValueError('Este PDF tiene contraseña; desbloquéalo legalmente antes de usarlo.')
PAGINAS = []
SIN_TEXTO = []
try:
    for n, pagina in enumerate(doc, start=1):
        texto = pagina.get_text('text', sort=True)
        texto = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', ' ', texto)
        texto = re.sub(r'[ \t]+', ' ', texto).strip()
        if not texto:
            SIN_TEXTO.append(n)
        else:
            PAGINAS.append((n, texto))
    TOTAL_PAGINAS = len(doc)
finally:
    doc.close()
PALABRAS = sum(len(re.findall(r'\b[\wÁÉÍÓÚÜÑáéíóúüñ]+\b', txt)) for n,txt in PAGINAS)
if not PAGINAS or PALABRAS < 30:
    raise ValueError('No se detectó texto suficiente: puede ser un PDF escaneado. Hace falta OCR, aún no incluido.')
print(f'✅ {NOMBRE_PDF}: {TOTAL_PAGINAS} páginas, {PALABRAS:,} palabras reconocidas como texto.')
if SIN_TEXTO:
    raise ValueError(f'⛔ Estas páginas están vacías o son escaneos: {SIN_TEXTO}. Para evitar omisiones, no generaremos el audiolibro. Usa un PDF con texto seleccionable.')
print('✅ Todas las páginas contienen texto. Ahora escucha una muestra.')

In [ ]:
# @title PASO 3 · Elegir voz y escuchar una muestra (pulsa ▶)
VOZ = 'Dora · femenina' # @param ['Dora · femenina', 'Alex · masculina', 'Santa · masculina']
VELOCIDAD = 1.0 # @param {type:'slider', min:0.8, max:1.3, step:0.05}
PAGINA_DE_MUESTRA = 1 # @param {type:'integer'}

from kokoro import KPipeline
from IPython.display import Audio, display
import numpy as np
import soundfile as sf
import torch

CODIGOS = {'Dora · femenina': 'ef_dora', 'Alex · masculina': 'em_alex', 'Santa · masculina': 'em_santa'}
CODIGO_VOZ = CODIGOS[VOZ]
if not 1 <= PAGINA_DE_MUESTRA <= TOTAL_PAGINAS:
    raise ValueError('El número de página para la muestra no existe en el PDF.')
texto_muestra = next((txt for n, txt in PAGINAS if n == PAGINA_DE_MUESTRA), None)
if not texto_muestra:
    raise ValueError('Esa página no contiene texto.')
texto_muestra = re.sub(r'\s+', ' ', texto_muestra)[:750]
if len(texto_muestra) > 450:
    corte = texto_muestra.rfind('. ', 380)
    texto_muestra = texto_muestra[:corte+1] if corte > 400 else texto_muestra

# El modelo se carga una vez y puede requerir descargar archivos al iniciar.
if 'PIPELINE' not in globals() or PIPELINE is None:
    PIPELINE = KPipeline(lang_code='e', device='cuda' if torch.cuda.is_available() else 'cpu')

def producir_trozos(texto, voz=CODIGO_VOZ, velocidad=VELOCIDAD):
    for _, _, audio in PIPELINE(texto, voice=voz, speed=velocidad):
        if audio is not None:
            trozo = np.asarray(audio, dtype=np.float32).ravel()
            if len(trozo):
                yield trozo

muestras = list(producir_trozos(texto_muestra))
if not muestras:
    raise RuntimeError('La voz no generó audio. No se realizará la conversión completa.')
muestra = np.concatenate(muestras)
if len(muestra)/24000 < 3:
    raise RuntimeError('La muestra parece demasiado corta; revisa el motor antes de continuar.')
print(f'▶ Muestra con la voz {VOZ}, {len(muestra)/24000:.1f} segundos. Escúchala ahora:')
display(Audio(muestra, rate=24000))
MUESTRA_OK = (CODIGO_VOZ, VELOCIDAD, NOMBRE_PDF)
print('Si te gusta la voz, continúa al paso 4. Si no, cambia la VOZ y vuelve a ejecutar este paso.')

In [ ]:
# @title PASO 4 · Convertir TODO el PDF y descargar MP3 (solo si te gustó la muestra)
if 'MUESTRA_OK' not in globals() or MUESTRA_OK != (CODIGOS[VOZ], VELOCIDAD, NOMBRE_PDF):
    raise RuntimeError('Primero genera la muestra del PASO 3 con esta voz, velocidad y PDF.')
import tempfile, subprocess, os, gc
import soundfile as sf

# Fragmentación conservadora por espacios para evitar textos demasiado largos.
def fragmentar(texto, maximo=750):
    texto = re.sub(r'\s+', ' ', texto).strip()
    if not texto: return []
    frases = re.split(r'(?<=[.!?;:])\s+', texto)
    partes = []
    actual = ''
    def agregar_largo(frase):
        while len(frase) > maximo:
            cortar = frase.rfind(' ', 0, maximo)
            if cortar < maximo // 2:
                cortar = maximo
            partes.append(frase[:cortar].strip())
            frase = frase[cortar:].strip()
        return frase
    for frase in frases:
        if len(frase) > maximo:
            if actual:
                partes.append(actual)
                actual = ''
            frase = agregar_largo(frase)
        if not frase:
            continue
        if actual and len(actual) + 1 + len(frase) > maximo:
            partes.append(actual)
            actual = frase
        else:
            actual = (actual + ' ' + frase).strip()
    if actual: partes.append(actual)
    return [p for p in partes if p]

nombre_base = unicodedata.normalize('NFKD', Path(NOMBRE_PDF).stem)
nombre_base = ''.join(c for c in nombre_base if not unicodedata.combining(c))
nombre_base = re.sub(r'[^a-zA-Z0-9_-]+', '_', nombre_base).strip('_')[:65] or 'unidad'
SALIDA = Path('/content') / f'Audiolibro_{nombre_base}.mp3'

with tempfile.TemporaryDirectory(prefix='audiolibro_') as tmp:
    wav = Path(tmp) / 'voz.wav'
    duracion_s = 0.0
    # Escribe el audio incrementalmente; evita mantener una hora de audio en memoria.
    with sf.SoundFile(str(wav), 'w', samplerate=24000, channels=1, subtype='PCM_16', format='WAV') as archivo_audio:
        for numero, texto in PAGINAS:
            trozos = fragmentar(texto)
            if not trozos:
                raise RuntimeError(f'La página {numero} está vacía, conversión cancelada.')
            segundos_pagina = 0.0
            for fragmento in trozos:
                fragmentos_audio = list(producir_trozos(fragmento))
                if not fragmentos_audio:
                    raise RuntimeError(f'Error de síntesis en página {numero}. No se entrega un archivo incompleto.')
                for a in fragmentos_audio:
                    archivo_audio.write(a)
                    segundos_pagina += len(a) / 24000
                del fragmentos_audio
            if segundos_pagina < 2:
                raise RuntimeError(f'No se generó suficiente audio para la página {numero}.')
            silencio = np.zeros(9600, dtype=np.float32) # 0.4s entre páginas
            archivo_audio.write(silencio)
            duracion_s += segundos_pagina + 0.4
            print(f'✅ Página {numero}/{TOTAL_PAGINAS} · {segundos_pagina/60:.1f} min de voz', flush=True)
    if duracion_s < PALABRAS * 60 / 420:
        raise RuntimeError('El audio final parece demasiado corto para el texto. No se entrega por precaución.')
    resultado = subprocess.run(
        ['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', str(wav),
         '-ac', '1', '-c:a', 'libmp3lame', '-b:a', '96k', str(SALIDA)],
        capture_output=True, text=True)
    if resultado.returncode != 0 or not SALIDA.exists() or SALIDA.stat().st_size < 5000:
        raise RuntimeError('Error creando MP3: ' + resultado.stderr[-400:])

print(f'🎉 MP3 listo: {SALIDA.name} · {duracion_s/60:.1f} minutos · {SALIDA.stat().st_size/1024/1024:.1f} MB')
print(f'📚 Páginas narradas: {len(PAGINAS)} de {TOTAL_PAGINAS}.')
print('Guárdalo en el teléfono para escuchar sin conexión.')
files.download(str(SALIDA))

### Limitaciones y comprobación de calidad
- El programa lee el **texto extraíble**, no inventa un resumen. No puede leer imágenes ni verificar por sí mismo la pronunciación de cada término.
- Si tu PDF tiene gráficos, fórmulas, diseños con varias columnas o páginas escaneadas, revisa primero que la lectura tenga sentido. Las páginas sin texto bloquean la conversión para evitar omisiones.
- Para voz más natural no se necesita una suscripción TTS por caracteres, pero la primera instalación puede tardar y Google Colab gratuito tiene recursos limitados.
- **Esta versión fue revisada estáticamente y probada con un motor de voz simulado; la voz real de Kokoro debe probarse en Colab** porque este entorno no permite descargar sus modelos. La vista previa del paso 3 permite valorar la voz antes de convertir todo.